# Shape: gate the outputs of a dbt run (Python notebook)

Run after a dbt job. Reads the dbt models' Delta tables, profiles them together, checks the
profile against a contract, optionally diffs it against a baseline, reads the dbt job's
`run_results.json` and `manifest.json`, and writes **one report** in which a failed dbt test and a
Shape finding about the same column appear together (`report.md`, `report.json`).

The exit value is `{models, rows, passed, dbtFailed, dbtTotal, violations, drifted, changes,
byColumn, artifactPath, reportPath, truncated, kernel}`. `passed` is false when a dbt test or model
failed, a contract rule was violated, or (with `failOnDrift`) the data drifted.

**[VERIFY]** where the dbt job writes `run_results.json` and `manifest.json` in OneLake, and how
the model tables of a Fabric Data Warehouse are read from a notebook (`tableRoot`), are checked
on the first live run (`integrations/fabric/RUNBOOK.md`, section 10).


In [ ]:
%%configure
{"vCores": 8}


In [ ]:
# Upload these wheels to the notebook's resources folder (Resources > builtin):
#   sqllocks_shape-0.9.1-py3-none-any.whl
#   sqllocks_shape_dbt-0.9.1-py3-none-any.whl
# They are installed by file path, so a package of the same name and version on PyPI can never be
# installed instead. For the Rust kernel, upload the Linux x86_64 platform wheel
#   sqllocks_shape-0.9.1-cp311-abi3-manylinux_2_28_x86_64.whl
# and put its path in place of the first wheel (the exit value reports the kernel).
# The Python runtime ships numpy 1.26.4 with pandas 2.1.4 and matplotlib 3.8.2, built for numpy 1.
# Shape needs numpy 2, and pandas 2.1.4 cannot load with numpy 2 (pyarrow then fails too), so
# pandas is upgraded to a 2.x built for numpy 2. pip then prints a dependency conflict for
# matplotlib 3.8.2: expected (Shape does not use it; do not plot in this session).
# In a pipeline run inline %pip is off by default; the notebook activity must pass the Boolean
# parameter _inlineInstallationEnabled = true (the shipped pipelines do). A Python notebook
# cannot attach an Environment, and %pip is not supported in High Concurrency mode or in a
# reference run (notebookutils.notebook.run).
%pip install "numpy>=2,<3" "pandas>=2.2.2,<3" builtin/sqllocks_shape-0.9.1-py3-none-any.whl builtin/sqllocks_shape_dbt-0.9.1-py3-none-any.whl


In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
# Paths are relative to the lakehouse Files/ folder, or absolute.
models = "orders,customers"   # the dbt models to profile: Delta tables <tableRoot>/<model>
tableRoot = "/lakehouse/default/Tables"  # where the model tables are (a OneLake path also works)
contractPath = ""         # optional multi-table contract: {"tables": {model: contract}}
baselinePath = ""         # optional earlier .shape artifact (of these models) to diff against
runResultsPath = ""       # run_results.json of the dbt run (the dbt job's output in OneLake)
manifestPath = ""         # manifest.json of the dbt run (names each test's model and column)
outputDir = "shape"       # artifacts go to Files/<outputDir>/dbt/<timestamp>/
failOnDrift = False       # True: drift against the baseline also fails the gate


In [ ]:
import json
import os
import warnings
from pathlib import Path

import shape
from shape.artifact.io import ArtifactNotVerifiedWarning
from shape.integrations.fabric.run_folder import claim_run_folder
from shape.kernel.dispatch import get_kernel
from shape_dbt import report as dbt_report

KERNEL = get_kernel().NAME
print(f"Shape {shape.__version__}, kernel: {KERNEL}")

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"
MAX_LISTED = 100  # keep the exit value small (well under 1 MB)


def _as_bool(value) -> bool:
    """Pipeline parameters may arrive as strings."""
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


def _resolve(path: str) -> str:
    return path if os.path.isabs(path) else f"{FILES}/{path}"


def load_baseline(path: str):
    """The baseline .shape (this notebook's own earlier artifact)."""
    with warnings.catch_warnings():
        # our own unsigned artifact: silence only "not signed" for this one load
        warnings.simplefilter("ignore", ArtifactNotVerifiedWarning)
        return shape.load(_resolve(path))


failOnDrift = _as_bool(failOnDrift)
safe_name = "dbt"
out_parent = f"{outputDir.strip('/')}/{safe_name}"  # one new folder per run is created below it


In [ ]:
import re

from deltalake import DeltaTable

names = [m.strip() for m in str(models).split(",") if m.strip()]
for name in names:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"{name!r} is not a model name (letters, digits and _)")
sources = {name: DeltaTable(f"{tableRoot.rstrip('/')}/{name}").to_pyarrow_table() for name in names}
total_rows = sum(t.num_rows for t in sources.values())
profile = shape.profile(sources, name="dbt")
print(f"Profiled {total_rows:,} rows in {len(sources)} models")


In [ ]:
check, drift = None, None

if contractPath:
    check = shape.check(profile, _resolve(contractPath)).to_dict()

if baselinePath:
    drift = shape.diff(load_baseline(baselinePath), profile).to_dict()

run_results = dbt_report.load_json(_resolve(runResultsPath), "run_results") if runResultsPath else None
manifest = dbt_report.load_json(_resolve(manifestPath), "manifest") if manifestPath else None
if run_results is None:
    run_results = {"results": []}  # no dbt run given: the report has Shape's findings only
report = dbt_report.build_report(
    run_results, manifest, check=check, drift=drift, fail_on_drift=failOnDrift
)
passed = report["ok"]
violations = (check or {}).get("violations", [])
changes = (drift or {}).get("changes", [])


In [ ]:
out_dir = claim_run_folder(Path(FILES) / out_parent)  # a folder no other run owns
out_rel = f"{out_parent}/{out_dir.name}"
shape.save(profile, str(out_dir / "dbt.shape"))
(out_dir / "dbt.html").write_text(profile.to_html(), encoding="utf-8")
(out_dir / "report.json").write_text(json.dumps(report, default=str), encoding="utf-8")
(out_dir / "report.md").write_text(dbt_report.render_markdown(report), encoding="utf-8")
artifact_path = f"{out_rel}/dbt.shape"  # relative to Files/, usable as baselinePath
report_path = f"{out_rel}/report.md"
print("Artifacts written to", out_dir)


In [ ]:
from IPython.display import HTML, Markdown, display

display(Markdown(dbt_report.render_markdown(report)))
display(HTML(profile.to_html()))


In [ ]:
result = {
    "models": {name: t.num_rows for name, t in sources.items()},
    "rows": total_rows,
    "passed": passed,
    "dbtFailed": report["summary"]["dbt_failed"],
    "dbtTotal": report["summary"]["dbt_total"],
    "violations": violations[:MAX_LISTED],
    "drifted": bool(changes),
    "changes": changes[:MAX_LISTED],
    "byColumn": sorted(report["by_column"])[:MAX_LISTED],
    "artifactPath": artifact_path,
    "reportPath": report_path,
    "truncated": len(violations) > MAX_LISTED or len(changes) > MAX_LISTED,
    "kernel": KERNEL,
}
print(json.dumps(result, indent=2, default=str)[:4000])


In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result, default=str))
